<a href="https://colab.research.google.com/github/JonasFanZ/114-2-Programing-Language/blob/main/%E3%80%8CHW3_%E5%BE%85%E8%BE%A6%E6%B8%85%E5%96%AE%E8%88%87%E7%95%AA%E8%8C%84%E9%90%98%E7%B4%80%E9%8C%84_ipynb%E3%80%8D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!pip -q install gspread gspread_dataframe google-auth google-auth-oauthlib google-auth-httplib2 \
               gradio pandas beautifulsoup4 google-generativeai python-dateutil

import os, time, uuid, re, json, datetime, math
from datetime import datetime as dt, timedelta
from dateutil.tz import gettz
import pandas as pd
import gradio as gr
import requests
from bs4 import BeautifulSoup

import google.generativeai as genai

# Google Auth & Sheets
from google.colab import auth
import gspread
from gspread_dataframe import set_with_dataframe, get_as_dataframe
from google.auth.transport.requests import Request
from google.oauth2 import service_account
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 嘗試獲取 API 金鑰，加入手動輸入備案防止 Colab UI 卡住
api_key = ""
try:
    from google.colab import userdata
    api_key = userdata.get('gemini')
    print("✅ 成功從 Colab Secrets 讀取 Gemini API 金鑰！")
except BaseException as e: # 捕捉包含 TimeoutException 在內的所有例外
    print(f"⚠️ Colab Secrets 讀取失敗：{e}")
    print("👇 系統無法自動抓取金鑰，請在下方對話框手動貼上：")
    import getpass
    api_key = getpass.getpass("🔑 請手動貼上你的 Gemini API 金鑰並按 Enter: ")

if api_key:
    genai.configure(api_key=api_key)
    print("✅ Gemini API 設定完成！")
else:
    print("⚠️ 警告：目前未設定 Gemini API 金鑰，AI 分析與排程功能可能會失敗。")

# 使用 gemini-2.5-flash
model = genai.GenerativeModel('gemini-3.1-flash-lite-preview')

SHEET_URL = "https://docs.google.com/spreadsheets/d/1fg56sy61HLyo9JMQ0cLd7aP3ajAmySb7OFYgphzPa6s/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"
TIMEZONE = "Asia/Taipei"

gsheets = gc.open_by_url(SHEET_URL)
sh = gsheets.worksheet(WORKSHEET_NAME).get_all_values()
df = pd.DataFrame(sh[1:], columns=sh[0])

def ensure_spreadsheet(name):
    try:
        sh = gc.open(name)
    except gspread.SpreadsheetNotFound:
        sh = gc.create(name)
    return sh

sh = ensure_spreadsheet(WORKSHEET_NAME)

def ensure_worksheet(sh, title, header):
    try:
        ws = sh.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=title, rows="1000", cols=str(len(header)+5))
        ws.update([header])
    data = ws.get_all_values()
    if not data or (data and data[0] != header):
        ws.clear()
        ws.update([header])
    return ws

TASKS_HEADER = [
    "id","task","status","priority","est_min","start_time","end_time",
    "actual_min","pomodoros","due_date","labels","notes",
    "created_at","updated_at","completed_at","planned_for"
]
LOGS_HEADER = [
    "log_id","task_id","phase","start_ts","end_ts","minutes","cycles","note"
]
CLIPS_HEADER = ["clip_id","url","selector","text","href","created_at","added_to_task"]

ws_tasks = ensure_worksheet(sh, "tasks", TASKS_HEADER)
ws_logs  = ensure_worksheet(sh, "pomodoro_logs", LOGS_HEADER)
ws_clips = ensure_worksheet(sh, "web_clips", CLIPS_HEADER)

def tznow():
    return dt.now(gettz(TIMEZONE))

def read_df(ws, header):
    df = get_as_dataframe(ws, evaluate_formulas=True, header=0)
    if df is None or df.empty:
        return pd.DataFrame(columns=header)
    df = df.fillna("")
    for c in header:
        if c not in df.columns:
            df[c] = ""

    # 強化資料型態的防呆機制，若 Google Sheet 為空字串則自動轉為 0
    for col in ["est_min", "actual_min", "pomodoros"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].replace(r'^\s*$', 0, regex=True), errors="coerce").fillna(0).astype(int)

    return df[header]

def write_df(ws, df, header):
    if df.empty:
        ws.clear()
        ws.update([header])
        return
    df_out = df.copy()
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)
    ws.clear()
    ws.update([header] + df_out.values.tolist())

def refresh_all():
    return (
        read_df(ws_tasks, TASKS_HEADER).copy(),
        read_df(ws_logs, LOGS_HEADER).copy(),
        read_df(ws_clips, CLIPS_HEADER).copy()
    )

tasks_df, logs_df, clips_df = refresh_all()

def get_display_tasks(df=None):
    if df is None:
        global tasks_df
        df = tasks_df

    if df.empty:
        return pd.DataFrame(columns=["狀態", "優先級", "任務名稱", "預估(分)", "實際(分)", "番茄數", "到期日"])

    display_df = df.copy()
    status_map = {"todo": "📝 待辦", "in-progress": "⏳ 進行中", "done": "✅ 完成"}
    display_df["status"] = display_df["status"].map(status_map).fillna(display_df["status"])
    priority_map = {"H": "🔴 高", "M": "🟡 中", "L": "🟢 低"}
    display_df["priority"] = display_df["priority"].map(priority_map).fillna(display_df["priority"])
    display_df = display_df[["status", "priority", "task", "est_min", "actual_min", "pomodoros", "due_date"]]
    display_df.columns = ["狀態", "優先級", "任務名稱", "預估(分)", "實際(分)", "番茄數", "到期日"]

    return display_df

def get_display_logs(df_logs=None):
    if df_logs is None:
        global logs_df
        df_logs = logs_df

    if df_logs.empty:
        return pd.DataFrame(columns=["日期時間", "任務名稱", "階段", "耗時(分)", "備註"])

    d_logs = df_logs.copy()
    global tasks_df
    if not tasks_df.empty:
        task_dict = dict(zip(tasks_df.id, tasks_df.task))
        d_logs["任務名稱"] = d_logs["task_id"].map(task_dict).fillna("未知任務 (可能已被刪除)")
    else:
        d_logs["任務名稱"] = d_logs["task_id"]

    try:
        d_logs["日期時間"] = pd.to_datetime(d_logs["start_ts"]).dt.strftime('%m/%d %H:%M')
    except:
        d_logs["日期時間"] = d_logs["start_ts"]

    phase_map = {"work": "🧠 專注", "break": "🍵 休息"}
    d_logs["階段"] = d_logs["phase"].map(phase_map).fillna(d_logs["phase"])
    d_logs = d_logs[["日期時間", "任務名稱", "階段", "minutes", "note"]]
    d_logs.columns = ["日期時間", "任務名稱", "階段", "耗時(分)", "備註"]
    d_logs = d_logs.sort_values(by="日期時間", ascending=False).reset_index(drop=True)
    return d_logs

def add_task(task, priority, est_min, due_date, labels, notes, planned_for):
    global tasks_df
    _now = tznow().isoformat()
    new = pd.DataFrame([{
        "id": str(uuid.uuid4())[:8],
        "task": task.strip(),
        "status": "todo",
        "priority": priority or "M",
        "est_min": int(est_min) if est_min else 25,
        "start_time": "",
        "end_time": "",
        "actual_min": 0,
        "pomodoros": 0,
        "due_date": due_date or "",
        "labels": labels or "",
        "notes": notes or "",
        "created_at": _now,
        "updated_at": _now,
        "completed_at": "",
        "planned_for": planned_for or ""
    }])
    tasks_df = pd.concat([tasks_df, new], ignore_index=True)
    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    return "✅ 已新增任務", "", "", ""

def update_task_status(task_id, new_status):
    global tasks_df
    idx = tasks_df.index[tasks_df["id"] == task_id]
    if len(idx)==0: return "⚠️ 找不到任務"
    i = idx[0]
    tasks_df.loc[i, "status"] = new_status
    tasks_df.loc[i, "updated_at"] = tznow().isoformat()
    if new_status == "done":
        if not tasks_df.loc[i, "completed_at"]:
            tasks_df.loc[i, "completed_at"] = tznow().isoformat()
    else:
        tasks_df.loc[i, "completed_at"] = ""
    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    return "✅ 狀態已更新"

def mark_done(task_id):
    return update_task_status(task_id, "done")

def delete_task(task_id):
    global tasks_df
    idx = tasks_df.index[tasks_df["id"] == task_id]
    if len(idx) == 0: return "⚠️ 找不到任務", gr.update()
    tasks_df = tasks_df.drop(idx).reset_index(drop=True)
    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    return "🗑️ 任務已成功刪除", gr.update(value=None)

def recalc_task_actuals(task_id):
    global tasks_df, logs_df
    work_logs = logs_df[(logs_df["task_id"]==task_id) & (logs_df["phase"]=="work")]
    total_min = work_logs["minutes"].astype(float).sum() if not work_logs.empty else 0
    # 🌟 修正：實際完成顆數以「每做滿25分鐘算1顆」(無條件捨去) 為準
    pomos = int(total_min // 25)
    idx = tasks_df.index[tasks_df["id"]==task_id]
    if len(idx)==0: return
    i = idx[0]
    tasks_df.loc[i,"actual_min"] = int(total_min)
    tasks_df.loc[i,"pomodoros"] = pomos
    tasks_df.loc[i,"updated_at"] = tznow().isoformat()

def list_task_choices():
    global tasks_df
    if tasks_df.empty: return []
    def row_label(r):
        status_map = {"todo": "待辦", "in-progress": "進行中", "done": "已完成"}
        s_name = status_map.get(r['status'], r['status'])
        return f"[{s_name}] (P:{r['priority']}) {r['task']}"
    return [(row_label(r), r["id"]) for _, r in tasks_df.iterrows()]

def show_task_details(task_id):
    global tasks_df
    if not task_id: return "> 💡 **提示**：請選擇上方的任務，即可在此查看其附帶的標籤與備註資訊。"
    idx = tasks_df.index[tasks_df["id"] == task_id]
    if len(idx) == 0: return "> ⚠️ **錯誤**：找不到此任務的詳細資料。"
    row = tasks_df.loc[idx[0]]
    lbls = row["labels"] if str(row["labels"]).strip() else "*(無標籤)*"
    nts = row["notes"] if str(row["notes"]).strip() else "*(無備註)*"
    return f"> **🏷️ 標籤：** {lbls}\n>\n> **📝 備註：** {nts}"

session_state = {
    "task_id": None,
    "work_elapsed": 0,
    "break_elapsed": 0,
    "work_remain": 25 * 60,
    "break_remain": 5 * 60,
    "phase": None,
    "running": False,
    "session_id": "",
    "start_ts": None
}

def format_time(seconds, state):
    mins, secs = divmod(int(seconds), 60)
    color = "#ff4757" if "work" in state else "#1e90ff"
    if "done" in state: color = "#2ed573"
    if state == "idle": color = "#a4b0be"
    return f"<h1 style='text-align: center; font-size: 6rem; font-weight: bold; color: {color}; font-family: monospace; margin: 10px 0;'>{mins:02d}:{secs:02d}</h1>"

def pause_timer():
    session_state["running"] = False
    return "⏸️ 計時已暫停"

def debug_fast_forward_j():
    global session_state
    if session_state["running"] and session_state["phase"] == "work":
        current_remain = session_state["work_remain"]
        if current_remain > 2:
            half = current_remain // 2
            skipped = current_remain - half
            session_state["work_remain"] = half
            session_state["work_elapsed"] += skipped
            print(f"[Debug J] 已快轉至一半時間，剩餘 {half} 秒")

def debug_fast_forward_k():
    global session_state
    if session_state["running"] and session_state["phase"] == "work":
        current_remain = session_state["work_remain"]
        if current_remain > 5:
            skipped = current_remain - 5
            session_state["work_remain"] = 5
            session_state["work_elapsed"] += skipped
            print(f"[Debug K] 已快轉至 5 秒")

def start_work_logic(task_id, cycles):
    global session_state
    if not task_id:
        yield "⚠️ 請先選擇任務", gr.update(), gr.update()
        return
    if session_state["task_id"] != task_id:
        session_state.update({"task_id": task_id, "work_elapsed": 0, "break_elapsed": 0, "work_remain": 25 * 60, "break_remain": 5 * 60, "start_ts": tznow().isoformat()})
    if session_state["work_remain"] <= 0: session_state["work_remain"] = 25 * 60

    session_state["phase"] = "work"
    session_state["running"] = True
    session_id = str(uuid.uuid4())
    session_state["session_id"] = session_id

    yield f"▶️ 已開始：專注工作（計時中...）", format_time(session_state["work_remain"], "work"), gr.update()
    while session_state["running"] and session_state["session_id"] == session_id and session_state["work_remain"] > 0:
        time.sleep(1)
        if not session_state["running"] or session_state["session_id"] != session_id: break
        session_state["work_remain"] -= 1
        session_state["work_elapsed"] += 1
        yield f"▶️ 已開始：專注工作（計時中...）", format_time(session_state["work_remain"], "work"), gr.update()

    if session_state["running"] and session_state["session_id"] == session_id and session_state["work_remain"] <= 0:
        session_state["running"] = False
        # 🌟 修改：倒數到 0 時，利用第三個返回值輸出目前的時間戳記，去觸發介面上的隱藏 auto_save_trigger
        yield "🎉 專注時間到！自動儲存紀錄...", format_time(0, "work_done"), str(time.time())

def start_break_logic(task_id, cycles):
    global session_state
    if not task_id:
        yield "⚠️ 請先選擇任務", gr.update(), gr.update()
        return
    if session_state["task_id"] != task_id:
        session_state.update({"task_id": task_id, "work_elapsed": 0, "break_elapsed": 0, "work_remain": 25 * 60, "break_remain": 5 * 60, "start_ts": tznow().isoformat()})
    if session_state["break_remain"] <= 0: session_state["break_remain"] = 5 * 60

    session_state["phase"] = "break"
    session_state["running"] = True
    session_id = str(uuid.uuid4())
    session_state["session_id"] = session_id

    yield f"▶️ 已開始：休息（計時中...）", format_time(session_state["break_remain"], "break"), gr.update()
    while session_state["running"] and session_state["session_id"] == session_id and session_state["break_remain"] > 0:
        time.sleep(1)
        if not session_state["running"] or session_state["session_id"] != session_id: break
        session_state["break_remain"] -= 1
        session_state["break_elapsed"] += 1
        yield f"▶️ 已開始：休息（計時中...）", format_time(session_state["break_remain"], "break"), gr.update()

    if session_state["running"] and session_state["session_id"] == session_id and session_state["break_remain"] <= 0:
        session_state["running"] = False
        # 🌟 修改：休息倒數到 0 時，同樣觸發隱藏儲存機關
        yield "🎉 休息結束！自動儲存紀錄...", format_time(0, "break_done"), str(time.time())

def end_phase_logic(task_id, note):
    global logs_df, tasks_df, session_state
    session_state["running"] = False
    session_state["session_id"] = ""

    if not session_state["task_id"]:
        return "⚠️ 尚未開始任何階段", format_time(25*60, "idle")

    new_logs = []
    end_ts = tznow().isoformat()
    if session_state["work_elapsed"] >= 60:
        minutes = round(session_state["work_elapsed"] / 60.0, 2)
        new_logs.append({
            "log_id": str(uuid.uuid4())[:8], "task_id": session_state["task_id"], "phase": "work",
            "start_ts": session_state["start_ts"], "end_ts": end_ts, "minutes": minutes, "cycles": 1, "note": note or ""
        })
    if session_state["break_elapsed"] >= 60:
        minutes = round(session_state["break_elapsed"] / 60.0, 2)
        new_logs.append({
            "log_id": str(uuid.uuid4())[:8], "task_id": session_state["task_id"], "phase": "break",
            "start_ts": session_state["start_ts"], "end_ts": end_ts, "minutes": minutes, "cycles": 1, "note": note or ""
        })

    if new_logs:
        logs_df = pd.concat([logs_df, pd.DataFrame(new_logs)], ignore_index=True)
        write_df(ws_logs, logs_df, LOGS_HEADER)
        recalc_task_actuals(session_state["task_id"])
        write_df(ws_tasks, tasks_df, TASKS_HEADER)
        work_m = round(session_state['work_elapsed']/60.0, 1)
        break_m = round(session_state['break_elapsed']/60.0, 1)
        msg = f"⏹️ 已儲存：累積專注 {work_m} 分 / 休息 {break_m} 分"
    else:
        msg = "⏹️ 已結束 (未滿 1 分鐘不予紀錄)"

    session_state.update({"task_id": None, "work_elapsed": 0, "break_elapsed": 0, "work_remain": 25 * 60, "break_remain": 5 * 60, "phase": None, "start_ts": None})
    return msg, format_time(25*60, "idle")

def get_this_week_moodle_tasks():
    global tasks_df
    if tasks_df.empty: return get_display_tasks(pd.DataFrame(columns=TASKS_HEADER))
    seven_days_ago = (tznow() - timedelta(days=7)).isoformat()
    mask = (tasks_df['labels'].fillna("").astype(str).str.contains('from:announcement', na=False)) & (tasks_df['created_at'].fillna("").astype(str) >= seven_days_ago)
    return get_display_tasks(tasks_df[mask])

def process_moodle_announcement(url, username, password):
    global tasks_df
    import time
    if not url.strip(): return "⚠️ 請輸入公告網址"
    if not username.strip() or not password.strip(): return "⚠️ 請輸入 Moodle 帳號與密碼"
    if not tasks_df.empty and tasks_df["notes"].fillna("").astype(str).str.contains(url, na=False).any():
        return f"⚠️ 發現重複：此公告網址已存在於任務清單中！"

    try:
        from urllib.parse import urlparse
        parsed_uri = urlparse(url)
        base_url = f"{parsed_uri.scheme}://{parsed_uri.netloc}"
        login_url = f"{base_url}/login/index.php"

        session = requests.Session()
        session.headers.update({"User-Agent": "Mozilla/5.0"})
        r_login = session.get(login_url, timeout=15)
        soup_login = BeautifulSoup(r_login.text, "html.parser")
        token_tag = soup_login.find("input", {"name": "logintoken"})
        logintoken = token_tag["value"] if token_tag else ""

        payload = {"username": username, "password": password, "logintoken": logintoken}
        r_post = session.post(login_url, data=payload, timeout=15)
        r_target = session.get(url, timeout=15)
        r_target.raise_for_status()

        soup = BeautifulSoup(r_target.text, "html.parser")
        text_content = soup.get_text(separator="\n", strip=True)[:2500]
    except Exception as e:
        return f"⚠️ 抓取網頁或登入失敗：{e}"

    prompt = f"""
    你是一位專業的課程助理。現在的系統日期是 {tznow().date().isoformat()}。
    請閱讀以下的「課程公告文字」，判斷其中是否包含任何「作業、考試、繳交事項或待辦事項」。
    要求：
    1. 如果沒有待辦事項，請設定 has_task 為 false。
    2. 如果有多個待辦事項，請將它們合併成一個任務。
    3. 判斷到期日 (due_date)，並轉換為 YYYY-MM-DD 格式。若無明確日期請留空字串 ""。
    4. 根據語氣判斷優先級 (priority)，填入 "H", "M", 或 "L"。
    5. 預估完成時間 (est_min)，以分鐘為單位整數（例如 60）。
    請務必只回傳合法的 JSON 格式：
    {{ "has_task": true/false, "task_name": "任務名稱", "due_date": "YYYY-MM-DD", "priority": "M", "est_min": 60, "reason": "簡單說明" }}
    公告內容：{text_content}
    """
    try:
        for attempt in range(3):
            try:
                response = model.generate_content(prompt, generation_config=genai.GenerationConfig(response_mime_type="application/json"))
                ai_result = json.loads(response.text)
                break
            except Exception as e:
                if attempt == 2: raise e
                time.sleep(2)
    except Exception as e:
        return f"⚠️ AI 判斷失敗：{e}"

    if not ai_result.get("has_task"): return f"ℹ️ AI 判斷這篇公告沒有待辦事項。({ai_result.get('reason', '')})"

    _now = tznow().isoformat()
    new_task = pd.DataFrame([{
        "id": str(uuid.uuid4())[:8], "task": f"[公告] {ai_result['task_name']}", "status": "todo",
        "priority": ai_result.get("priority", "M"), "est_min": int(ai_result.get("est_min", 30)),
        "start_time": "", "end_time": "", "actual_min": 0, "pomodoros": 0, "due_date": ai_result.get("due_date", ""),
        "labels": "from:announcement", "notes": f"AI判斷理由：{ai_result.get('reason','')}\n來源網址：{url}",
        "created_at": _now, "updated_at": _now, "completed_at": "", "planned_for": ""
    }])
    tasks_df = pd.concat([tasks_df, new_task], ignore_index=True)
    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    return f"✅ 已成功從公告萃取任務：{ai_result['task_name']} (預估 {ai_result.get('est_min')} 分鐘)"

def generate_today_plan():
    global tasks_df
    today = tznow().date().isoformat()
    cand = tasks_df[tasks_df["status"] != "done"].copy()
    if cand.empty: return "📭 恭喜！目前試算表中沒有任何未完成的任務哦！"

    pr_order = {"H":0, "M":1, "L":2}
    cand["p_ord"] = cand["priority"].map(pr_order).fillna(3)
    cand = cand.sort_values(["p_ord","est_min"], ascending=[True, True])

    buckets = {"morning": [], "afternoon": [], "evening": []}
    for i, (_, r) in enumerate(cand.iterrows()):
        if i % 3 == 0: buckets["morning"].append(r)
        elif i % 3 == 1: buckets["afternoon"].append(r)
        else: buckets["evening"].append(r)

    def sec_md(name, rows):
        if not rows: return f"### {name.title()}\n（無）\n"
        lines = [f"### {name.title()}"]
        for r in rows: lines.append(f"- [{r['id']}] {r['task']}（預估 {int(r['est_min'])} 分，P:{r['priority']}）")
        return "\n".join(lines) + "\n"
    rule_md = sec_md("morning", buckets["morning"]) + "\n" + sec_md("afternoon", buckets["afternoon"]) + "\n" + sec_md("evening", buckets["evening"])

    try:
        sys_prompt = "你是一位任務規劃助理。請把輸入的所有待辦任務排成三段：morning、afternoon、evening，並給出每段的重點、順序、每項的時間預估與備註。回傳以 Markdown 條列，格式：\n### Morning\n- [任務名稱]（預估 xx 分）— 備註\n...\n### Afternoon\n...\n### Evening\n...\n"
        items = [{"id": r["id"], "task": r["task"], "est_min": int(r["est_min"]), "priority": r["priority"], "labels": r["labels"]} for _, r in cand.iterrows()]
        user_content = json.dumps({"today": today, "tasks": items}, ensure_ascii=False)
        resp = model.generate_content(sys_prompt + "\n\n" + user_content)
        return resp.text.strip()
    except Exception as e:
        return (f"⚠️ Gemini 失敗或超時：{e}\n\n🤖 已為您改用傳統規則式排程：\n---\n" + rule_md).strip()

def today_summary():
    global tasks_df
    today = tznow().date().isoformat()
    cand_all = tasks_df[(tasks_df["status"] != "done") | (tasks_df["completed_at"].fillna("").astype(str).str.startswith(today))]
    total = len(cand_all)
    done_n = len(cand_all[cand_all["status"] == "done"])
    rate = (done_n/total*100) if total>0 else 0
    return f"📅 待辦任務總庫存：{total} 項；✅ 今日消滅：{done_n} 項；📈 總達成率：{rate:.1f}%"

def crawl(url, selector, mode, limit):
    try:
        resp = requests.get(url, timeout=15, headers={"User-Agent":"Mozilla/5.0"})
        resp.raise_for_status()
    except Exception as e: return pd.DataFrame(columns=CLIPS_HEADER), f"⚠️ 請求失敗：{e}"
    soup = BeautifulSoup(resp.text, "html.parser")
    nodes = soup.select(selector)
    rows = []
    for i, n in enumerate(nodes[:int(limit) if limit else 20]):
        text = n.get_text(strip=True) if mode in ("text","both") else ""
        href = n.get("href") if mode in ("href","both") else ""
        if href and href.startswith("/"):
            from urllib.parse import urljoin
            href = urljoin(url, href)
        rows.append({"clip_id": str(uuid.uuid4())[:8], "url": url, "selector": selector, "text": text, "href": href, "created_at": tznow().isoformat(), "added_to_task": ""})
    return pd.DataFrame(rows, columns=CLIPS_HEADER), f"✅ 擷取 {len(rows)} 筆"

def add_clips_as_tasks(clip_ids, default_priority, est_min):
    global clips_df, tasks_df
    if not clip_ids: return "⚠️ 請先勾選要加入的爬蟲項目"
    sel = clips_df[clips_df["clip_id"].isin(clip_ids)]
    _now = tznow().isoformat()
    new_tasks = []
    for _, r in sel.iterrows():
        new_tasks.append({
            "id": str(uuid.uuid4())[:8], "task": (r["text"] or r["href"] or "（未命名）")[:120], "status": "todo",
            "priority": default_priority or "M", "est_min": int(est_min) if est_min else 25, "start_time": "", "end_time": "",
            "actual_min": 0, "pomodoros": 0, "due_date": "", "labels": "from:crawler",
            "notes": f"來源：{r['url']}\n連結：{r['href']}", "created_at": _now, "updated_at": _now, "completed_at": "", "planned_for": ""
        })
    if new_tasks:
        tasks_df = pd.concat([tasks_df, pd.DataFrame(new_tasks)], ignore_index=True)
        clips_df.loc[clips_df["clip_id"].isin(clip_ids), "added_to_task"] = "yes"
        write_df(ws_tasks, tasks_df, TASKS_HEADER)
        write_df(ws_clips, clips_df, CLIPS_HEADER)
        return f"✅ 已加入 {len(new_tasks)} 項為任務"
    return "⚠️ 無可加入項目"

def update_task_info(task_id):
    global tasks_df
    if not task_id:
        return 1, "> 💡 **提示**：請先選擇上方任務，來查看目前進度..."

    idx = tasks_df.index[tasks_df["id"] == task_id]
    if len(idx) == 0:
        return 1, "> ⚠️ **錯誤**：找不到任務資料"

    row = tasks_df.loc[idx[0]]
    try:
        val_est = str(row["est_min"]).strip()
        est_min = int(float(val_est)) if val_est else 0
    except: est_min = 0

    try:
        val_pomo = str(row["pomodoros"]).strip()
        actual_pomos = int(float(val_pomo)) if val_pomo else 0
    except: actual_pomos = 0

    est_pomos = math.ceil(est_min / 25.0) if est_min > 0 else 1
    remain = max(0, est_pomos - actual_pomos)
    progress_md = f"**📌 目標：** {row['task']} &nbsp;&nbsp;｜&nbsp;&nbsp; **🎯 預估總數：** {est_pomos} 顆 &nbsp;&nbsp;｜&nbsp;&nbsp; **✅ 已完成：** {actual_pomos} 顆 &nbsp;&nbsp;｜&nbsp;&nbsp; **🚀 距離目標還剩：** <span style='color: #ff4757; font-weight: bold;'>{remain} 顆</span>"
    return est_pomos, progress_md

# 🌟 重構：整合所有刷新邏輯，徹底解決「選單重置」與「進度不更新」的問題
def _refresh(current_task_choice=None, current_sel_task=None):
    global tasks_df, logs_df, clips_df
    tasks_df, logs_df, clips_df = refresh_all()
    task_choices_list = list_task_choices()

    # 🌟 防呆檢查：如果剛才選的任務還在清單裡，就保留它，治好下拉選單失憶症！
    valid_ids = [c[1] for c in task_choices_list]
    tc_val = current_task_choice if current_task_choice in valid_ids else None
    st_val = current_sel_task if current_sel_task in valid_ids else None

    # 🌟 自動更新：在刷新資料庫後，直接重新算出最新的備註卡與番茄鐘進度！
    task_details_md = show_task_details(tc_val)
    _, task_progress_md = update_task_info(st_val)

    return (
        get_display_tasks(tasks_df),
        get_display_logs(logs_df),
        clips_df,
        gr.update(choices=task_choices_list, value=tc_val),
        today_summary(),
        gr.update(choices=task_choices_list, value=st_val),
        get_this_week_moodle_tasks(),
        task_progress_md,
        task_details_md
    )

with gr.Blocks(theme=gr.themes.Soft(), title="待辦清單＋番茄鐘＋AI 計畫") as demo:

    with gr.Group():
        with gr.Row(variant="panel"):
            with gr.Column(scale=4):
                gr.Markdown("# 🍅 待辦清單與番茄鐘整合系統\n*(Google Sheet + Gradio + Crawler + AI 課程公告判讀)*")
                out_summary = gr.Markdown(today_summary())
            with gr.Column(scale=1, min_width=150):
                btn_refresh = gr.Button("🔄 重新整理資料", variant="primary")

    with gr.Tab("📝 任務管理 (Tasks)"):
        gr.Markdown("### 🗂️ 目前所有任務清單")
        grid_tasks = gr.Dataframe(value=get_display_tasks(tasks_df), interactive=False)

        with gr.Row():
            with gr.Column(scale=1):
                with gr.Group():
                    gr.Markdown("### ➕ 新增任務")
                    task = gr.Textbox(label="任務名稱", placeholder="寫 HW3 報告 / 修正 SQL / …")
                    with gr.Row():
                        priority = gr.Dropdown(["H","M","L"], value="M", label="優先級")
                        est_min = gr.Number(value=25, label="預估時間（分鐘）", precision=0)
                    with gr.Row():
                        due_date = gr.Textbox(label="到期日（YYYY-MM-DD）")
                        planned_for = gr.Dropdown(["","today","tomorrow"], value="", label="規劃歸屬")

                    with gr.Accordion("📂 更多設定 (標籤與備註)", open=False):
                        labels = gr.Textbox(label="標籤（逗號分隔，可空白）")
                        notes = gr.Textbox(label="備註（可空白）")

                    btn_add = gr.Button("新增任務", variant="primary")
                    msg_add = gr.Markdown()

            with gr.Column(scale=1):
                with gr.Group():
                    gr.Markdown("### ✏️ 任務詳情與狀態更新")
                    task_choice = gr.Dropdown(choices=list_task_choices(), label="選擇要查看或更新的任務")

                    task_details_card = gr.Markdown("> 💡 **提示**：請選擇上方的任務，即可在此查看其附帶的標籤與備註資訊。")

                    new_status = gr.Dropdown(["todo","in-progress","done"], value="in-progress", label="更新狀態")
                    with gr.Row():
                        btn_update = gr.Button("更新狀態")
                        btn_done = gr.Button("✅ 快速標記完成", variant="primary")
                        btn_delete = gr.Button("🗑️ 刪除任務", variant="stop")
                    msg_update = gr.Markdown()

    with gr.Tab("🍅 番茄鐘 (Pomodoro)"):
        with gr.Group():
            gr.Markdown("### 🎯 選擇目標與進度")
            sel_task = gr.Dropdown(choices=list_task_choices(), label="選擇今日要執行的任務")
            task_progress = gr.Markdown("> 💡 **提示**：請先選擇上方任務，來查看目前進度...")
            cycles = gr.Number(value=1, visible=False)

        with gr.Group():
            gr.Markdown("### ⏱️ 專注計時器")
            timer_display = gr.HTML("<h1 style='text-align: center; font-size: 6rem; font-weight: bold; color: #a4b0be; font-family: monospace; margin: 10px 0;'>25:00</h1>")
            msg_pomo = gr.Markdown(value="尚未開始計時。", label="系統狀態")

            with gr.Row():
                btn_start_work = gr.Button("▶️ 開始專注", variant="primary", scale=2)
                btn_pause = gr.Button("⏸️ 暫停", scale=1)
                btn_start_break = gr.Button("🍵 開始休息", scale=2)

            with gr.Row():
                btn_debug_j = gr.Button("⏩ 快轉至一半時間 (測試用)", size="sm")
                btn_debug_k = gr.Button("⏭️ 快轉至最後 5 秒 (測試用)", size="sm")

            with gr.Row(variant="panel"):
                note_phase = gr.Textbox(label="備註 (選填，紀錄剛才的專注或休息重點)", scale=4)
                btn_end_phase = gr.Button("⏹️ 結束計時並儲存紀錄", variant="stop", scale=1)

                # 🌟 新增：隱藏的自動儲存觸發器
                auto_save_trigger = gr.Textbox(visible=False)

        with gr.Group():
            gr.Markdown("### 📊 歷史番茄鐘紀錄")
            grid_logs = gr.Dataframe(value=get_display_logs(logs_df), interactive=False)

    with gr.Tab("🎓 課程公告分析 (Moodle)"):
        gr.Markdown("輸入您的 Moodle 帳號密碼與公告網址，系統會在背景自動登入並抓取公告文字給 AI 分析！")
        with gr.Row():
            with gr.Column(scale=1):
                moodle_url = gr.Textbox(label="公告網址 URL", placeholder="https://moodle.xxx.edu.tw/mod/forum/discuss.php?d=1234")
                with gr.Row():
                    moodle_user = gr.Textbox(label="Moodle 帳號 (通常是學號)", scale=1)
                    moodle_pwd = gr.Textbox(label="Moodle 密碼", type="password", scale=1)
                btn_process_moodle = gr.Button("🤖 模擬登入擷取並分析任務", variant="primary")
                msg_moodle = gr.Markdown()
            with gr.Column(scale=1):
                gr.Markdown("### 📅 本週從公告產生的任務清單")
                grid_moodle_tasks = gr.Dataframe(value=get_this_week_moodle_tasks(), interactive=False)

    with gr.Tab("🤖 智能計畫 (AI Plan)"):
        with gr.Group():
            gr.Markdown("讓 AI 幫你把**目前試算表裡所有還沒做完的任務**排成 **morning / afternoon / evening** 三段讀書計畫。")
            btn_plan = gr.Button("🧠 一鍵為我安排未完成的任務", variant="primary")
        with gr.Group():
            out_plan = gr.Markdown()

    with gr.Tab("🕷️ 網頁爬蟲 (Crawler)"):
        with gr.Row():
            with gr.Column(scale=1):
                with gr.Group():
                    gr.Markdown("### 1. 爬取設定")
                    url = gr.Textbox(label="目標 URL", placeholder="https://example.com")
                    selector = gr.Textbox(label="CSS Selector", placeholder="a.news-item / h2.title / div.card a")
                    with gr.Row():
                        mode = gr.Radio(["text","href","both"], value="text", label="擷取內容")
                        limit = gr.Number(value=20, precision=0, label="最多擷取幾筆")
                    btn_crawl = gr.Button("開始擷取", variant="primary")
                    msg_crawl = gr.Markdown()

            with gr.Column(scale=1):
                with gr.Group():
                    gr.Markdown("### 2. 轉換為任務")
                    clip_ids = gr.Textbox(label="要加入任務的 clip_id（多個以逗號分隔）")
                    with gr.Row():
                        default_priority = gr.Dropdown(["H","M","L"], value="L", label="新增任務優先級")
                        clip_est = gr.Number(value=25, precision=0, label="新增任務預估分鐘")
                    btn_add_clips = gr.Button("➕ 將勾選的項目加入為任務", variant="primary")
                    msg_add_clips = gr.Markdown()

        gr.Markdown("### 擷取結果清單")
        grid_clips = gr.Dataframe(value=clips_df, interactive=True)

    with gr.Tab("Summary"):
        btn_summary = gr.Button("📊 重新計算任務消耗率")
        out_summary2 = gr.Markdown()

    # ==========================================
    # 🌟 統一更新管線：綁定所有事件至新的 _refresh
    # ==========================================
    # 定義統一的 Refresh Inputs 和 Outputs
    refresh_inputs = [task_choice, sel_task]
    refresh_outputs = [
        grid_tasks, grid_logs, grid_clips,
        task_choice, out_summary, sel_task,
        grid_moodle_tasks, task_progress, task_details_card
    ]

    btn_refresh.click(_refresh, inputs=refresh_inputs, outputs=refresh_outputs)

    btn_add.click(
        add_task, inputs=[task, priority, est_min, due_date, labels, notes, planned_for], outputs=[msg_add, task, labels, notes]
    ).then(_refresh, inputs=refresh_inputs, outputs=refresh_outputs)

    task_choice.change(fn=show_task_details, inputs=[task_choice], outputs=[task_details_card])

    btn_update.click(
        update_task_status, inputs=[task_choice, new_status], outputs=[msg_update]
    ).then(_refresh, inputs=refresh_inputs, outputs=refresh_outputs)

    btn_done.click(
        mark_done, inputs=[task_choice], outputs=[msg_update]
    ).then(_refresh, inputs=refresh_inputs, outputs=refresh_outputs)

    btn_delete.click(
        delete_task, inputs=[task_choice], outputs=[msg_update, task_choice]
    ).then(_refresh, inputs=refresh_inputs, outputs=refresh_outputs)

    sel_task.change(fn=update_task_info, inputs=[sel_task], outputs=[cycles, task_progress])

    # 🌟 將自動儲存觸發器綁定至倒數結束的回傳值
    btn_start_work.click(start_work_logic, inputs=[sel_task, cycles], outputs=[msg_pomo, timer_display, auto_save_trigger])
    btn_pause.click(pause_timer, outputs=[msg_pomo])
    btn_start_break.click(start_break_logic, inputs=[sel_task, cycles], outputs=[msg_pomo, timer_display, auto_save_trigger])

    btn_debug_j.click(fn=debug_fast_forward_j)
    btn_debug_k.click(fn=debug_fast_forward_k)

    # 🌟 修改：不管你是手動按下按鈕，還是隱藏的觸發器發生改變，都會執行同一套完美收尾與存檔流程！
    btn_end_phase.click(
        end_phase_logic, inputs=[sel_task, note_phase], outputs=[msg_pomo, timer_display]
    ).then(
        lambda: "", outputs=[note_phase]
    ).then(
        _refresh, inputs=refresh_inputs, outputs=refresh_outputs
    )

    auto_save_trigger.change(
        end_phase_logic, inputs=[sel_task, note_phase], outputs=[msg_pomo, timer_display]
    ).then(
        lambda: "", outputs=[note_phase]
    ).then(
        _refresh, inputs=refresh_inputs, outputs=refresh_outputs
    )

    btn_process_moodle.click(
        fn=lambda: "⏳ **爬蟲與 AI 判讀進行中，請稍候約 5~15 秒...**", inputs=[], outputs=[msg_moodle]
    ).then(
        process_moodle_announcement, inputs=[moodle_url, moodle_user, moodle_pwd], outputs=[msg_moodle]
    ).then(_refresh, inputs=refresh_inputs, outputs=refresh_outputs)

    btn_plan.click(
        fn=lambda: "⏳ **AI 正在為您排程中，請稍候約 5~15 秒...**", inputs=[], outputs=[out_plan]
    ).then(
        generate_today_plan, inputs=[], outputs=[out_plan]
    )

    def _crawl_and_save(u, s, m, l):
        df, msg = crawl(u, s, m, l)
        global clips_df
        if not df.empty:
            clips_df = pd.concat([clips_df, df], ignore_index=True)
            write_df(ws_clips, clips_df, CLIPS_HEADER)
        return msg, clips_df

    btn_crawl.click(_crawl_and_save, inputs=[url, selector, mode, limit], outputs=[msg_crawl, grid_clips])

    def _add_clips(clip_ids_str, pr, est):
        ids = [c.strip() for c in (clip_ids_str or "").split(",") if c.strip()]
        return add_clips_as_tasks(ids, pr, est)

    btn_add_clips.click(
        _add_clips, inputs=[clip_ids, default_priority, clip_est], outputs=[msg_add_clips]
    ).then(_refresh, inputs=refresh_inputs, outputs=refresh_outputs)

    btn_summary.click(today_summary, outputs=[out_summary2])

demo.queue().launch(debug=True)

✅ 成功從 Colab Secrets 讀取 Gemini API 金鑰！
✅ Gemini API 設定完成！


/tmp/ipykernel_2071/3158674418.py:600: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="待辦清單＋番茄鐘＋AI 計畫") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://992a31e08575691754.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[Debug J] 已快轉至一半時間，剩餘 743 秒
[Debug K] 已快轉至 5 秒


/tmp/ipykernel_2071/3158674418.py:379: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  logs_df = pd.concat([logs_df, pd.DataFrame(new_logs)], ignore_index=True)


[Debug K] 已快轉至 5 秒
[Debug K] 已快轉至 5 秒
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://992a31e08575691754.gradio.live
